# AETHER: Lunar PSR Enhancement
**Chandrayaan-2 OHRC Zero-DCE Training Pipeline**

This notebook trains a Zero-DCE (Zero-Reference Deep Curve Estimation) model to enhance Permanently Shadowed Regions (PSRs) on the Moon. It uses a 2-phase training strategy:
1. **Phase 1 (Supervised Adaptation)**: Train on synthetically darkened sunlit terrain to learn basic feature recovery and noise suppression.
2. **Phase 2 (Self-Supervised Fine-Tuning)**: Train on actual dark PSR patches using no-reference loss functions (Spatial Consistency, Exposure Control, Illumination Smoothness).

**Hardware**: Run this on GPU (T4 x2 or P100).
**Dataset**: Make sure the `chandrayaan-2-ohrc-lunar-psrs` dataset is attached to this notebook.

In [ ]:
!pip install piq pyiqa -q
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Define Dataset Path (Update this if your dataset name differs)
DATA_DIR = '/kaggle/input/chandrayaan-2-ohrc-lunar-psrs/patches'

### 1. Model Architecture (Zero-DCE)

In [ ]:
class ZeroDCE(nn.Module):
    def __init__(self, channels=32, n_iter=8):
        super().__init__()
        self.n_iter = n_iter
        self.conv1 = nn.Conv2d(1, channels, 3, padding=1, padding_mode='replicate')
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv3 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv4 = nn.Conv2d(channels, channels, 3, padding=1, padding_mode='replicate')
        self.conv5 = nn.Conv2d(channels*2, channels, 3, padding=1, padding_mode='replicate')
        self.conv6 = nn.Conv2d(channels*2, channels, 3, padding=1, padding_mode='replicate')
        self.conv7 = nn.Conv2d(channels*2, 8, 3, padding=1, padding_mode='replicate')
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(x2))
        x4 = self.relu(self.conv4(x3))
        x5 = self.relu(self.conv5(torch.cat([x3, x4], dim=1)))
        x6 = self.relu(self.conv6(torch.cat([x2, x5], dim=1)))
        A = torch.tanh(self.conv7(torch.cat([x1, x6], dim=1)))
        enhanced = x
        for i in range(self.n_iter):
            A_i = A[:, i:i+1, :, :]
            enhanced = enhanced + A_i * (torch.pow(enhanced, 2) - enhanced)
        return enhanced, A

### 2. Loss Functions

In [ ]:
class ZeroDCELoss(nn.Module):
    def __init__(self, w_spa=1.0, w_exp=10.0, w_tv=200.0, E=0.6, patch_size=16):
        super().__init__()
        self.w_spa = w_spa
        self.w_exp = w_exp
        self.w_tv = w_tv
        self.E = E
        self.pool = nn.AvgPool2d(patch_size, stride=patch_size)
        kernel = torch.zeros(4, 1, 3, 3)
        kernel[0, 0, 1, 0] = -1; kernel[0, 0, 1, 1] = 1
        kernel[1, 0, 1, 2] = -1; kernel[1, 0, 1, 1] = 1
        kernel[2, 0, 0, 1] = -1; kernel[2, 0, 1, 1] = 1
        kernel[3, 0, 2, 1] = -1; kernel[3, 0, 1, 1] = 1
        self.register_buffer('kernel', kernel)

    def spatial_loss(self, enhanced, original):
        e_pool = F.avg_pool2d(enhanced, 4)
        o_pool = F.avg_pool2d(original, 4)
        e_grad = F.conv2d(e_pool, self.kernel, padding=1)
        o_grad = F.conv2d(o_pool, self.kernel, padding=1)
        return torch.mean((e_grad - o_grad) ** 2)

    def exposure_loss(self, enhanced):
        e_pool = self.pool(enhanced)
        return torch.mean((e_pool - self.E) ** 2)

    def tv_loss(self, A):
        tv_h = torch.mean((A[:, :, 1:, :] - A[:, :, :-1, :]) ** 2)
        tv_w = torch.mean((A[:, :, :, 1:] - A[:, :, :, :-1]) ** 2)
        return tv_h + tv_w

    def forward(self, original, enhanced, A):
        spa = self.spatial_loss(enhanced, original)
        exp = self.exposure_loss(enhanced)
        tv = self.tv_loss(A)
        total_loss = self.w_spa * spa + self.w_exp * exp + self.w_tv * tv
        return total_loss, {'spa': spa.item(), 'exp': exp.item(), 'tv': tv.item()}

### 3. Datasets

In [ ]:
class SyntheticPSRDataset(Dataset):
    def __init__(self, npy_path):
        self.data = np.load(npy_path)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        clean = self.data[idx]
        gamma = np.random.uniform(0.02, 0.15)
        dark = clean * gamma
        alpha, beta = 0.01, 0.001
        noise_var = alpha * dark + beta
        noise = np.random.normal(0, np.sqrt(noise_var), dark.shape)
        dark_noisy = np.clip(dark + noise, 0, 1)
        return torch.from_numpy(dark_noisy).unsqueeze(0).float(), torch.from_numpy(clean).unsqueeze(0).float()

class RealPSRDataset(Dataset):
    def __init__(self, npy_path):
        self.data = np.load(npy_path)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.from_numpy(self.data[idx]).unsqueeze(0).float()

def show_batch(batch, title):
    x, y = batch if isinstance(batch, (list, tuple)) else (batch, batch)
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    fig.suptitle(title, fontsize=16)
    for i in range(4):
        if i >= len(x): break
        axes[0, i].imshow(x[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[0, i].axis('off')
        axes[0, i].set_title(f'Input (Mean: {x[i].mean():.3f})')
        axes[1, i].imshow(y[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[1, i].axis('off')
        axes[1, i].set_title('Target')
    plt.tight_layout()
    plt.show()

### 4. Training Loop

In [ ]:
def train_phase1(model, dataloader, epochs=20, lr=1e-4):
    print('--- Starting Phase 1: Synthetic Supervised Pre-training ---')
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    l1_loss = nn.L1Loss()
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
        for x, y_clean in pbar:
            x, y_clean = x.to(device), y_clean.to(device)
            optimizer.zero_grad()
            y_pred, _ = model(x)
            loss = l1_loss(y_pred, y_clean)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({'L1 Loss': f'{loss.item():.4f}'})
    torch.save(model.state_dict(), 'zerodce_phase1.pth')
    print('Phase 1 complete. Model saved.')

def train_phase2(model, dataloader, epochs=15, lr=1e-5):
    print('--- Starting Phase 2: Real PSR Self-Supervised Fine-tuning ---')
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    zero_loss = ZeroDCELoss().to(device)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        pbar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}')
        for x in pbar:
            x = x.to(device)
            optimizer.zero_grad()
            y_pred, A = model(x)
            loss, metrics = zero_loss(x, y_pred, A)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({'Total Loss': f'{loss.item():.4f}', 'Exp': f'{metrics["exp"]:.4f}'})
    torch.save(model.state_dict(), 'zerodce_phase2_final.pth')
    print('Phase 2 complete. Model saved.')

### 5. Execute Training Pipeline

In [ ]:
model = ZeroDCE().to(device)
print('Loading data...')
try:
    ds_syn = SyntheticPSRDataset(f'{DATA_DIR}/sunlit/train.npy')
    dl_syn = DataLoader(ds_syn, batch_size=32, shuffle=True, num_workers=2)
    ds_real = RealPSRDataset(f'{DATA_DIR}/psr/train.npy')
    dl_real = DataLoader(ds_real, batch_size=32, shuffle=True, num_workers=2)
    print(f'Loaded {len(ds_syn)} synthetic pairs and {len(ds_real)} real PSR patches.')
    sample_batch = next(iter(dl_syn))
    show_batch(sample_batch, 'Synthetic Training Data (Phase 1)')
except FileNotFoundError as e:
    print(f'Error: {e}. Please ensure the dataset is attached to this notebook.')

train_phase1(model, dl_syn, epochs=15, lr=1e-4)
train_phase2(model, dl_real, epochs=10, lr=1e-5)

### 6. Visualize Results on Real PSR Data

In [ ]:
model.eval()
val_ds = RealPSRDataset(f'{DATA_DIR}/psr/val.npy')
val_dl = DataLoader(val_ds, batch_size=4, shuffle=True)
with torch.no_grad():
    x_test = next(iter(val_dl)).to(device)
    y_test, _ = model(x_test)
    x_cpu = x_test.cpu().numpy()
    y_cpu = y_test.cpu().numpy()
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    fig.suptitle('Real PSR Enhancement Results', fontsize=16)
    for i in range(4):
        x_vis = np.clip(x_cpu[i, 0] * 5, 0, 1) 
        axes[0, i].imshow(x_vis, cmap='gray', vmin=0, vmax=1)
        axes[0, i].axis('off')
        axes[0, i].set_title(f'Raw PSR (Mean: {x_cpu[i].mean():.3f})')
        axes[1, i].imshow(y_cpu[i, 0], cmap='gray', vmin=0, vmax=1)
        axes[1, i].axis('off')
        axes[1, i].set_title(f'Zero-DCE (Mean: {y_cpu[i].mean():.3f})')
    plt.tight_layout()
    plt.show()